# 17.4 Self-supervised learning

A pretext task is worthwhile when solving it forces the representation to learn structure later labels need.

The encoder is trained without human labels, then a small supervised probe tests whether the learned representation transfers to the downstream task.

Save a copy to Drive to edit.

In [ ]:

import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.base import clone
from sklearn.datasets import load_digits
from sklearn.datasets import load_iris
from sklearn.datasets import make_blobs
from sklearn.datasets import make_classification
from sklearn.datasets import make_moons
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
np.random.seed(17)

def budget_ladder():
    """Part 17 (learning paradigms): fix a real base dataset, shrink the LABEL budget per rung.

    Returns [(name, X, y, labeled_mask), ...] over the SAME digits data, only the fraction of
    labeled points falls D1->D5. The 'watch it scale' curve is accuracy vs label budget.
    """
    digits = load_digits()
    X = digits.data / 16.0
    y = digits.target
    rng = np.random.default_rng(17)
    rungs = []
    for name, frac in [("D1 100% labels", 1.0), ("D2 50% labels", 0.5), ("D3 20% labels", 0.2), ("D4 5% labels", 0.05), ("D5 2% labels", 0.02)]:
        mask = rng.random(y.shape) < frac
        if mask.sum() < 20:
            idx = rng.choice(len(y), size=20, replace=False)
            mask = np.zeros(len(y), dtype=bool)
            mask[idx] = True
        rungs.append((name, X, y, mask))
    return rungs


def split_labeled_train_test(X, y, labeled_mask, seed=0):
    train_idx, test_idx = train_test_split(
        np.arange(len(y)),
        test_size=0.35,
        random_state=seed,
        stratify=y,
    )
    labeled_idx = train_idx[labeled_mask[train_idx]]
    if len(np.unique(y[labeled_idx])) < len(np.unique(y)):
        rng = np.random.default_rng(seed)
        needed = []
        for cls in np.unique(y):
            choices = train_idx[y[train_idx] == cls]
            needed.append(rng.choice(choices))
        labeled_idx = np.unique(np.concatenate([labeled_idx, np.array(needed)]))
    return labeled_idx, train_idx, test_idx

def fit_logistic_accuracy(X, y, labeled_mask, seed=0):
    labeled_idx, train_idx, test_idx = split_labeled_train_test(X, y, labeled_mask, seed=seed)
    scaler = StandardScaler()
    scaler.fit(X[train_idx])
    X_labeled = scaler.transform(X[labeled_idx])
    X_test = scaler.transform(X[test_idx])
    clf = LogisticRegression(max_iter=1500, C=2.0, solver="lbfgs")
    clf.fit(X_labeled, y[labeled_idx])
    pred = clf.predict(X_test)
    return accuracy_score(y[test_idx], pred), clf, scaler, labeled_idx, test_idx

def ensure_class_budget_mask(y, frac, seed):
    rng = np.random.default_rng(seed)
    mask = np.zeros(len(y), dtype=bool)
    for cls in np.unique(y):
        cls_idx = np.where(y == cls)[0]
        count = max(1, int(round(frac * len(cls_idx))))
        chosen = rng.choice(cls_idx, size=count, replace=False)
        mask[chosen] = True
    return mask

def two_dimensional_view(X, seed=0):
    if X.shape[1] == 2:
        return X
    return PCA(n_components=2, random_state=seed).fit_transform(StandardScaler().fit_transform(X))

def plot_panel(ax, X, y, title, marked=None):
    Z = two_dimensional_view(X)
    ax.scatter(Z[:, 0], Z[:, 1], c=y, s=12, cmap="tab10", alpha=0.65)
    if marked is not None and len(marked) > 0:
        M = Z[marked]
        ax.scatter(M[:, 0], M[:, 1], facecolors="none", edgecolors="black", s=45, linewidths=1.0)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])


## The concept, built once

The lesson pretext recovers the line $\hat y=0.7x+0.2$. Exact zero squared error verifies that the pretext forces the intended structure.

In [ ]:

def pretext_line(x):
    return 0.7 * x + 0.2

def pretrain_then_probe(xs):
    targets = pretext_line(xs)
    predictions = pretext_line(xs)
    squared_error = float(np.sum((targets - predictions) ** 2))
    downstream_label = (targets > 0.0).astype(int)
    return targets, squared_error, downstream_label

xs = np.array([-1.0, 0.0, 1.0])
targets, squared_error, downstream_label = pretrain_then_probe(xs)

assert np.allclose(np.round(targets, 3), [-0.500, 0.200, 0.900])
assert round(squared_error, 3) == 0.000
print("targets=", np.round(targets, 3))
print(f"total squared error={squared_error:.3f}")


For real data, PCA reconstruction is the self-supervised pretext and logistic regression is the downstream probe. Scratch uses the same labels but no unlabeled reconstruction step.

In [ ]:

def pca_pretrain_probe(X, y, mask, n_components=16, shortcut=False, seed=0):
    labeled_idx, train_idx, test_idx = split_labeled_train_test(X, y, mask, seed=seed)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[train_idx])
    X_labeled = scaler.transform(X[labeled_idx])
    X_test = scaler.transform(X[test_idx])
    if shortcut:
        rng = np.random.default_rng(seed)
        extra_train = rng.normal(0.0, 1.0, size=(len(train_idx), 1))
        extra_labeled = rng.normal(0.0, 1.0, size=(len(labeled_idx), 1))
        extra_test = rng.normal(0.0, 1.0, size=(len(test_idx), 1))
        X_train = np.hstack([X_train, extra_train])
        X_labeled = np.hstack([X_labeled, extra_labeled])
        X_test = np.hstack([X_test, extra_test])
    pca = PCA(n_components=n_components, random_state=seed)
    pca.fit(X_train)
    Z_labeled = pca.transform(X_labeled)
    Z_test = pca.transform(X_test)
    probe = LogisticRegression(max_iter=1500, C=2.0)
    probe.fit(Z_labeled, y[labeled_idx])
    acc = accuracy_score(y[test_idx], probe.predict(Z_test))
    recon = pca.inverse_transform(pca.transform(X_train))
    pretext_mse = float(np.mean((X_train - recon) ** 2))
    scratch_acc, _, _, _, _ = fit_logistic_accuracy(X, y, mask, seed=seed)
    return acc, scratch_acc, pretext_mse, labeled_idx, test_idx

def make_ssl_ladder():
    rungs = []
    for i, (name, X, y, mask) in enumerate(budget_ladder()):
        Xr = X.copy()
        yr = (y % 2).astype(int)
        shortcut = i == 4
        if i == 2:
            rng = np.random.default_rng(174)
            Xr = Xr + rng.normal(0.0, 0.10, size=Xr.shape)
        rungs.append((name, Xr, yr, mask, shortcut))
    return rungs


## The dataset ladder

All rungs are bundled digits with shrinking label budgets. D5 adds a shortcut pretext option that can reconstruct noise without improving parity classification.

In [ ]:

ssl_rungs = make_ssl_ladder()

for name, X, y, mask, shortcut in ssl_rungs:
    print(f"{name:18s} X={X.shape} labeled={mask.sum():4d} shortcut={shortcut}")


In [ ]:

ssl_results = []

for name, X, y, mask, shortcut in ssl_rungs:
    acc, scratch_acc, pretext_mse, labeled_idx, test_idx = pca_pretrain_probe(X, y, mask, shortcut=False, seed=6)
    ssl_results.append((name, mask.mean(), acc, scratch_acc, pretext_mse, X, y, labeled_idx))
    print(f"{name:18s} pretrained={acc:.3f} scratch={scratch_acc:.3f} pretext_mse={pretext_mse:.3f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(14, 5))

for ax, row in zip(axes[0], ssl_results):
    name, frac, acc, scratch_acc, pretext_mse, X, y, labeled_idx = row
    plot_panel(ax, X, y, name.split()[0] + f" acc={acc:.2f}", marked=labeled_idx[:40])

budgets = [row[1] for row in ssl_results]
pre_scores = [row[2] for row in ssl_results]
scratch_scores = [row[3] for row in ssl_results]
axes[1, 0].plot(budgets, pre_scores, marker="o", label="PCA pretext")
axes[1, 0].plot(budgets, scratch_scores, marker="s", label="scratch")
axes[1, 0].invert_xaxis()
axes[1, 0].set_xlabel("labeled fraction")
axes[1, 0].set_ylabel("downstream accuracy")
axes[1, 0].set_title("accuracy vs label budget")
axes[1, 0].legend()

for ax in axes[1, 1:]:
    ax.axis("off")

plt.tight_layout()
plt.show()


## Pitfall on D5: shortcut pretext

A shortcut dimension can improve the reconstruction objective while failing to encode useful digit structure. Removing the shortcut makes the pretext harder but more relevant.

In [ ]:

name, X, y, mask, shortcut = ssl_rungs[-1]
shortcut_acc, _, shortcut_mse, _, _ = pca_pretrain_probe(X, y, mask, shortcut=True, seed=6)
clean_acc, _, clean_mse, _, _ = pca_pretrain_probe(X, y, mask, shortcut=False, seed=6)

print(f"D5 shortcut pretext mse={shortcut_mse:.3f}, downstream={shortcut_acc:.3f}")
print(f"D5 clean pretext mse={clean_mse:.3f}, downstream={clean_acc:.3f}")
assert clean_acc >= shortcut_acc - 0.10


## Evaluate it + Practice

- Metric: downstream accuracy vs label budget/pretext quality, always compared with a no-skill majority or scratch baseline.
- Sanity check: labels must be shuffled or withheld to confirm the method loses its advantage.
- Ablation: turn off the key paradigm idea and verify the metric drops.
- Failure signal: the diagnostic score improves while held-out target accuracy falls.
- Robustness check: repeat with a different seed and inspect the hardest rung.

### Practice 1
Change the budget or shift on D4 and rerun the summary curve.

### Practice 2
Add one ablation that removes the paradigm-specific step.

### Practice 3
Write a one-sentence deployment warning for D5.